# Task 1.4 - Two ML Models with Scikit-learn

This notebook contains two supervised learning workflows:
1. **Regression** - predicting a numeric value (house prices)
2. **Classification** - predicting a category (Titanic survival)

Each follows the full workflow: split → preprocess → train baseline → evaluate → inspect errors.

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("All libraries imported successfully!")

All libraries imported successfully!


## Part 1: Regression - Predicting House Prices

We'll use the California Housing dataset (built into scikit-learn) to predict 
median house value based on features like income, rooms, and location.

In [13]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df_housing = housing.frame

print("Shape:", df_housing.shape)
print("\nColumns:", df_housing.columns.tolist())
df_housing.head()

Shape: (20640, 9)

Columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [14]:
# Separate features (X) and target (y)
X = df_housing.drop(columns=['MedHouseVal'])
y = df_housing['MedHouseVal']

# Split into train and test sets BEFORE any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (16512, 8)
Test set size: (4128, 8)


In [15]:
# Build a pipeline: scale features, then train a Linear Regression baseline
regression_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

# Train the baseline model
regression_pipeline.fit(X_train, y_train)

print("Baseline Linear Regression model trained successfully!")

Baseline Linear Regression model trained successfully!


In [16]:
# Make predictions on the test set
y_pred = regression_pipeline.predict(X_test)

# Evaluate using MAE, RMSE, and R²
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- Regression Model Evaluation ---")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

--- Regression Model Evaluation ---
MAE (Mean Absolute Error): 0.5332
RMSE (Root Mean Squared Error): 0.7456
R² Score: 0.5758


**Interpretation:** The baseline Linear Regression model explains about 58% 
of the variance in house prices (R²=0.58), with an average prediction error 
of roughly $53,000. This is a reasonable starting point, but the model likely 
struggles with non-linear relationships in the data (e.g., location effects 
captured by latitude/longitude aren't naturally linear). A more complex model 
(like a tree-based method) could likely improve on this baseline.

In [17]:
# Create a comparison table of actual vs predicted values
comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Error': y_test.values - y_pred
})

# Show the 5 worst predictions (largest absolute error)
comparison['AbsError'] = comparison['Error'].abs()
worst_predictions = comparison.sort_values('AbsError', ascending=False).head(5)

print("5 worst predictions:")
worst_predictions

5 worst predictions:


,Actual,Predicted,Error,AbsError
3722,1.62500,11.500331,-9.875331,9.875331
1649,5.00001,0.851622,4.148388,4.148388
1140,5.00001,1.115227,3.884783,3.884783
872,5.00001,1.319954,3.680056,3.680056
3710,4.50000,0.867216,3.632784,3.632784


**Observation on worst predictions:** The largest error (row 3722) shows the 
model predicting ~$1.15M for a house actually valued at only $162,500 — a 
massive overprediction, likely caused by an unusual combination of features 
(e.g., high income area with an atypically low-value property).

More importantly, several of the worst predictions have an actual value of 
exactly 5.00001 — this is the dataset's known price cap at $500,000. Houses 
truly worth more were recorded at this ceiling, which confuses the model 
since it's trying to learn a linear relationship for values that were 
artificially truncated. This is a documented limitation of this dataset, 
not a flaw in our modeling approach.

## Part 2: Classification - Predicting Titanic Survival

We'll reuse the cleaned Titanic dataset from Task 1.2 to predict whether a 
passenger survived (binary classification: 0 = did not survive, 1 = survived).

In [18]:
# Load the cleaned Titanic dataset from Task 1.2
df_titanic = pd.read_csv("../Task-1.2-Data-Cleaning-EDA/dataset/titanic_cleaned.csv")

print("Shape:", df_titanic.shape)
df_titanic.head()

Shape: (891, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


In [19]:
# Select relevant features for predicting survival
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

X_titanic = df_titanic[features]
y_titanic = df_titanic[target]

# Identify which columns are numeric vs categorical (text)
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Pclass', 'Sex', 'Embarked']

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("\nTarget value counts:")
print(y_titanic.value_counts())

Numeric features: ['Age', 'SibSp', 'Parch', 'Fare']
Categorical features: ['Pclass', 'Sex', 'Embarked']

Target value counts:
Survived
0    549
1    342
Name: count, dtype: int64


In [20]:
# Split into train/test BEFORE preprocessing
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42, stratify=y_titanic
)

print("Training set size:", X_train_t.shape)
print("Test set size:", X_test_t.shape)

# Preprocessing: scale numeric features, one-hot encode categorical features
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features)
])

# Build the classification pipeline
classification_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

# Train the baseline model
classification_pipeline.fit(X_train_t, y_train_t)
print("\nBaseline Logistic Regression model trained successfully!")

Training set size: (712, 7)
Test set size: (179, 7)

Baseline Logistic Regression model trained successfully!


In [21]:
# Make predictions on the test set
y_pred_t = classification_pipeline.predict(X_test_t)

# Evaluate using multiple metrics (important for imbalanced classes)
accuracy = accuracy_score(y_test_t, y_pred_t)
precision = precision_score(y_test_t, y_pred_t)
recall = recall_score(y_test_t, y_pred_t)
f1 = f1_score(y_test_t, y_pred_t)
conf_matrix = confusion_matrix(y_test_t, y_pred_t)

print("--- Classification Model Evaluation ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"\nConfusion Matrix:\n{conf_matrix}")

--- Classification Model Evaluation ---
Accuracy: 0.8045
Precision: 0.7931
Recall: 0.6667
F1 Score: 0.7244

Confusion Matrix:
[[98 12]
 [23 46]]


**Interpretation:** The model achieves 80% accuracy, which is meaningfully 
better than the 61% baseline of always predicting "did not survive." 

However, recall (0.667) is notably lower than precision (0.793) — the model 
misses 23 out of 69 actual survivors (false negatives), more than the 12 
false positives. This suggests the model is somewhat conservative about 
predicting survival, possibly because survival was influenced by factors 
like specific cabin location or lifeboat assignment that aren't captured 
in our available features (Pclass, Sex, Age, fare, etc.).

Given the historical context uncovered in Task 1.2 (strong gender and class 
effects on survival), the model likely performs best on clear cases (e.g., 
wealthy female passengers) and struggles on borderline cases where multiple 
factors conflict.

In [22]:
# Look at specific misclassified passengers
results = X_test_t.copy()
results['Actual'] = y_test_t.values
results['Predicted'] = y_pred_t
misclassified = results[results['Actual'] != results['Predicted']]

print(f"Total misclassified: {len(misclassified)} out of {len(results)}")
misclassified.head(10)

Total misclassified: 35 out of 179


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Actual,Predicted
553,3,male,22.0,0,0,7.2250,C,1,0
559,3,female,36.0,1,0,17.4000,S,1,0
279,3,female,35.0,1,1,20.2500,S,1,0
712,1,male,48.0,1,0,52.0000,S,1,0
455,3,male,29.0,0,0,7.8958,C,1,0
65,3,male,28.0,1,1,15.2458,C,1,0
489,3,male,9.0,1,1,15.9000,S,1,0
869,3,male,4.0,1,1,11.1333,S,1,0
165,3,male,9.0,0,2,20.5250,S,1,0
297,1,female,2.0,1,2,151.5500,S,0,1


**Observation on misclassified examples:** Of the 35 misclassified passengers, 
the majority are 3rd class males who actually survived but were predicted 
not to. This suggests the model has strongly learned the general pattern 
"male + 3rd class = did not survive" (which holds true on average, as shown 
in Task 1.2), but this generalization fails for exceptions — such as young 
boys (ages 4 and 9 in our examples) who may have been prioritized due to 
age despite their class and gender.

This highlights a real limitation: our model captures broad group-level 
survival patterns well, but struggles with individual exceptions where 
age, family size, or other unmeasured factors (like specific cabin location) 
override the dominant trend.

## Summary

| Model | Type | Key Metric | Result |
|-------|------|-----------|--------|
| Linear Regression | Regression | R² Score | 0.576 |
| Logistic Regression | Classification | Accuracy | 0.805 |

Both models used a proper train/test split (before preprocessing), a 
Pipeline for consistent transformations, and a baseline-first approach 
before any tuning. Both were evaluated with multiple relevant metrics 
rather than a single number, and errors were inspected to understand 
each model's specific weaknesses - overprediction on price-capped houses 
for regression, and missed exceptions to gender/class patterns for 
classification.